In [2]:
import pandas as pd
from sklearn.cluster import KMeans

In [3]:
## import generated features

features = pd.read_csv('features_output_1.csv.gz', compression='gzip')
##remove interpolated_bandwidth column
features = features.drop(columns=['interpolated_bandwidth'])
labels = pd.read_csv('data/40mhz/original_labels/brc-2002_086400-01-output_true_labels.csv.gz', compression='gzip')

## combine features and labels into a single dataframe without key
data = pd.concat([features, labels], axis=1)

In [4]:
data

,amplitude_max,width,fall_time,rise_time,area_under_curve,ascendent_slope,descendent_slope,pulse_symmetry_mse,max_curvature,pulse_kurtosis,pulse_center_of_mass,energy_above_threshold,Unnamed: 0,class,class_detailed,0
0,50,3,4,0,97.5,0.0,-11.500000,0.0,3.0,-1.298306,1.083969,4598.0,25,eMuonic,eMuonicN,eMuonicN_n_P
1,1,1,1,0,0.5,0.0,-1.000000,0.0,NaN,NaN,0.000000,1.0,39,eElectromagnetic,ePhotons,ePhotons
2,3,2,1,1,4.0,1.0,-3.000000,2.0,-4.0,NaN,0.600000,13.0,54,eElectromagnetic,ePhotons,ePhotons
3,44,2,6,0,56.5,0.0,-7.333333,0.0,5.0,NaN,0.662791,2665.0,55,eMuonic,eMuonicP,eMuonicN_n_P
4,20,3,3,0,40.5,0.0,-6.333333,0.0,-7.0,-1.275191,0.862745,882.0,64,eElectromagnetic,ePhotons,ePhotons
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959437,84,2,5,0,108.0,0.0,-15.800000,0.0,0.0,NaN,0.629630,9972.0,4666399,eMuonic,eMuonicP,eMuonicN_n_P
959438,2,1,1,0,1.0,0.0,-2.000000,0.0,NaN,NaN,0.000000,4.0,4666402,eElectromagnetic,ePhotons,ePhotons
959439,2,3,2,1,4.5,1.0,-1.000000,0.5,-2.0,-1.371901,1.200000,8.0,4666411,eElectromagnetic,ePositrons,eElectrons_n_ePositrons
959440,4,1,2,0,3.0,0.0,-2.000000,0.0,NaN,NaN,0.333333,16.0,4666412,eElectromagnetic,ePhotons,ePhotons


In [10]:
## show uniques in last column
data.iloc[:,-1:].nunique()

cluster    6
dtype: int64

In [5]:

## drop nan values
print(f'Number of samples before dropping NaNs: {data.shape[0]}')

data = data.dropna()
labels = labels.dropna()

print(f'Number of samples after dropping NaNs: {data.shape[0]}')

## min max scaling
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data[features.columns] = scaler.fit_transform(data[features.columns])

Number of samples before dropping NaNs: 959442
Number of samples after dropping NaNs: 142977


/tmp/ipykernel_41157/4065031971.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[features.columns] = scaler.fit_transform(data[features.columns])


In [6]:
kmeans = KMeans(n_clusters=6, init='random')
kmeans.fit(data[features.columns])
clusters = kmeans.labels_
data['cluster'] = clusters

/tmp/ipykernel_41157/1894677609.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['cluster'] = clusters


In [8]:
## compare clusters with true labels
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(data['class'], data['cluster'])
print(f'Adjusted Rand Index between KMeans clusters and true labels: {ari}')

Adjusted Rand Index between KMeans clusters and true labels: 0.11758669095189707


In [12]:
## print class distribution per cluster but classes are in the last column
for cluster in range(6):
    cluster_data = data[data['cluster'] == cluster]
    class_counts = cluster_data.iloc[:,-2:].value_counts()
    print(f'Cluster {cluster}:')
    print(class_counts)

Cluster 0:
0                        cluster
eMuonicN_n_P             0          32377
ePhotons                 0           8274
eElectrons_n_ePositrons  0           4093
eNeutrons                0            331
eOthers                  0            330
eHadronic                0              3
Name: count, dtype: int64
Cluster 1:
0                        cluster
ePhotons                 1          19483
eMuonicN_n_P             1           5962
eElectrons_n_ePositrons  1           4582
eNeutrons                1            888
eOthers                  1            220
eHadronic                1              1
Name: count, dtype: int64
Cluster 2:
0                        cluster
ePhotons                 2          12874
eElectrons_n_ePositrons  2           3279
eMuonicN_n_P             2           2986
eNeutrons                2            809
eOthers                  2            162
Name: count, dtype: int64
Cluster 3:
0                        cluster
ePhotons                 3      